In [6]:
import pandas as pd,numpy as np,json
import getpass,os
from langchain.chat_models import init_chat_model
from ai_patterns_mining import parse_json_safe

/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
data = pd.read_csv('./clustered_dataset/umap_clustered_dataset.csv')

In [8]:
data.head()

,Pattern Name,Problem,Context,Solution,Result,Uses,cluster
0,External Knowledge Augmentation,Large Language Models (LLMs) are bounded by th...,LLMs relying on fixed and parametric knowledge...,Augment LLMs with the capability to access ext...,LLMs can surpass traditional knowledge limitat...,"Accessing contemporary information, retrieving...",0
1,Modular Knowledge Consolidation Pipeline,Raw evidence retrieved from external sources i...,Implementing Retrieval-Augmented Generation (R...,Decompose the knowledge consolidation process ...,"Provides the LLM with high-quality, relevant, ...","Open-domain question answering, information-se...",0
2,Parameter Extraction and Tool Invocation,"After selecting a tool, the LLM needs to corre...",A selected tool with its detailed documentatio...,The LLM extracts required parameters (content ...,Successful and accurate execution of external ...,"Making API calls, executing code, querying dat...",1
3,Domain-Specific Tool Integration,"LLMs, trained on general knowledge, often exhi...",LLMs needing to perform tasks requiring deep e...,Employ specific external tools like online cal...,"Mitigates the expertise gap in LLMs, enhancing...","Performing complex calculations, solving equat...",1
4,Tool Retrieval and Selection,"After task planning, efficiently and accuratel...","Subquestions generated from task planning, and...",Employ a two-step approach: 1) **Retriever-bas...,Efficiently narrows down the pool of potential...,"Choosing the right API, function, or external ...",1


In [9]:
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")

llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai", temperature=0)

In [10]:
summarizing_prompt = """\
You are an expert AI PATTERN ANALYZER. Your task is to analyze the patterns in the dataset and provide a concise summary for each cluster.

Your output should be a JSON. Each element should contain the following fields:
- cluster_id: The unique identifier of the cluster.
- short_name: A brief, descriptive name for the cluster (max 5 words).
- description: A detailed description of the common characteristics and themes of the patterns in the cluster (make it short).
- sub_patterns: A list of all pattern names that belong to the cluster.

Here is the dataset you need to analyze:
{data}
"""

In [11]:
def save_log_summarization(cluster_id, group, summary,is_final=False):
    with open('./logs/clusters/summarization_1.txt', 'a') as f:

        f.write(f"Cluster {cluster_id}\n")
        f.write("="*100 + "\n\n")
        
        # Original Patterns (pretty JSON)
        f.write("Original Patterns:\n")
        f.write(json.dumps(group.to_dict(orient="records"), indent=4, ensure_ascii=False))
        f.write("\n\n")
        
        # Summarized Patterns (pretty JSON)
        f.write("Summarized Patterns:\n")
        f.write(json.dumps(summary, indent=4, ensure_ascii=False))
        f.write("\n\n")
        if is_final:
            f.write("="*100 + "\n")
        else:
            f.write("-"*100 + "\n")

In [12]:
def summarize_cluster(cluster_df):
    cluster_data = cluster_df.to_dict(orient='records')
    _prompt = summarizing_prompt
    prompt = _prompt.format(data=cluster_data)
    response = llm.invoke(prompt)
    res_json =  parse_json_safe(response.content,delimiter='{}')
    save_log_summarization(cluster_df['cluster'].iloc[0], cluster_df, res_json, is_final=True)
    return res_json

In [13]:
summarizations = []
for cluster_id, group in data.groupby('cluster'):
    print(f"Processing Cluster {cluster_id} with {len(group)} patterns...")
    summary = summarize_cluster(group)
    print(f" - {len(summary)} summarized patterns:")
    print(" - ", summary)
    summarizations.append(summary)

Processing Cluster 0 with 2 patterns...
 - 4 summarized patterns:
 -  {'cluster_id': 0, 'short_name': 'External Knowledge Grounding', 'description': 'This cluster addresses LLM limitations regarding outdated or incomplete knowledge by integrating and processing external, dynamic information. Patterns focus on accessing real-time data sources, structuring raw evidence through multi-stage pipelines, and refining knowledge to improve factual accuracy, relevance, and mitigate hallucinations in LLM outputs.', 'sub_patterns': ['External Knowledge Augmentation', 'Modular Knowledge Consolidation Pipeline']}
Processing Cluster 1 with 5 patterns...
 - 4 summarized patterns:
 -  {'cluster_id': 1, 'short_name': 'LLM External Tool Interaction', 'description': 'These patterns describe how LLMs integrate, select, invoke, and orchestrate external tools to augment their capabilities, overcome limitations, and solve complex, multi-step tasks.', 'sub_patterns': ['Parameter Extraction and Tool Invocation'

In [14]:
summarizations

[{'cluster_id': 0,
  'short_name': 'External Knowledge Grounding',
  'description': 'This cluster addresses LLM limitations regarding outdated or incomplete knowledge by integrating and processing external, dynamic information. Patterns focus on accessing real-time data sources, structuring raw evidence through multi-stage pipelines, and refining knowledge to improve factual accuracy, relevance, and mitigate hallucinations in LLM outputs.',
  'sub_patterns': ['External Knowledge Augmentation',
   'Modular Knowledge Consolidation Pipeline']},
 {'cluster_id': 1,
  'short_name': 'LLM External Tool Interaction',
  'description': 'These patterns describe how LLMs integrate, select, invoke, and orchestrate external tools to augment their capabilities, overcome limitations, and solve complex, multi-step tasks.',
  'sub_patterns': ['Parameter Extraction and Tool Invocation',
   'Domain-Specific Tool Integration',
   'Tool Retrieval and Selection',
   'Tool Use / Tool Augmentation',
   'Tool Or

In [15]:
pd.DataFrame(summarizations).to_json('./clustered_dataset/cluster_summarizations.json', index=False, orient='records', force_ascii=False,indent=2,lines=True)